<div dir="rtl" style="text-align:right">
<h1 style="text-align:right">مدل واقعاً کدام متن را دیده است؟</h1>
<p style="text-align:right">درس 64 از 92 · چرا <bdi dir="ltr">Prompt</bdi> ناشناخته یا خیلی بلند دردسر می‌سازد؟ · <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">57-prompts</code></p>
<p style="text-align:right"><a target="_self" href="http://127.0.0.1:8000/part-09/chapter-03/57-prompts.html">📖 بازگشت به همین درس</a></p>
<p style="text-align:right">شناسه‌های ناشناخته و <bdi dir="ltr">Context</bdi> بریده‌شده را پیش از قضاوت دربارهٔ خروجی بررسی کنید.</p><p style="text-align:right">پیش‌نیاز: <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">CharacterTokenizer</code>، شناسهٔ ناشناخته و <bdi dir="ltr">Context Window</bdi>.</p>
<p style="text-align:right">زمان یادگیری درس همراه با همین دفتر: حدود ۶۰–۱۱۰ دقیقه. زمان دفتر دوباره به زمان درس اضافه نمی‌شود؛ نصب و تمرین اختیاری جداست.</p>
<p style="text-align:right">این دفتر نیمهٔ عملی درس است. مثال‌ها آمادهٔ اجرا هستند؛ دو <bdi dir="ltr">Cell</bdi> با برچسب <bdi dir="ltr">TODO</bdi> را خودتان کامل کنید. پیام <bdi dir="ltr">INCOMPLETE</bdi> یعنی هنوز چیزی ننوشته‌اید، نه اینکه پاسخ درست است. جواب مرجع در این دفتر پنهان نشده است.</p>
<p style="text-align:right">از بالا به پایین اجرا کنید. پس از تغییر هر تابع، <bdi dir="ltr">Cell</bdi> آن و سپس <bdi dir="ltr">Cell</bdi> آزمون را دوباره اجرا کنید. برای بررسی نهایی، از منوی <code style="direction:ltr;text-align:left;unicode-bidi:isolate">Kernel → Restart Kernel and Run All Cells</code> استفاده کنید.</p>
</div>

In [ ]:
from pathlib import Path
import os
import sys

project_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "mini_gpt").is_dir() and (p / "book_src").is_dir()), None)
if project_root is None:
    raise RuntimeError("Extract the complete learning project; open this notebook inside it.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print("Python:", sys.executable)
print("Project:", project_root)

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">قبل از اجرا، پیش‌بینی کنید</h2>
<p style="text-align:right">اگر دو <bdi dir="ltr">Prompt</bdi> بلند فقط در ابتدای حذف‌شده فرق کنند، آخرین ورودی مدل در تولید <bdi dir="ltr">Greedy</bdi> چه تفاوتی دارد؟</p>
</div>

<div dir="rtl" style="text-align:right"><p style="text-align:right">پیش‌بینی من: …</p></div>

In [ ]:
import torch
from mini_gpt.tokenizer import CharacterTokenizer
from mini_gpt.config import ModelConfig
from mini_gpt.model import MiniGPT
torch.set_num_threads(1)
torch.manual_seed(7)
tokenizer = CharacterTokenizer.from_text('مدل زبان ')
model = MiniGPT(ModelConfig(tokenizer.vocab_size,5,8,2,1,0.0)).eval()
text = 'مدل X زبان مدل'
ids = torch.tensor([tokenizer.encode(text)])
print('all IDs:',ids.tolist(),'effective IDs:',ids[:,-5:].tolist())

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">این بار شما کد بنویسید</h2>
<p style="text-align:right"><code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">effective_ids(tokenizer, text, limit)</code> دو <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">list</code> برگرداند: شناسه‌های کامل متن و حداکثر <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">limit</code> شناسهٔ آخر. متن غیرخالی و <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">limit</code> مثبت است؛ <bdi dir="ltr">Vocabulary</bdi> را تغییر ندهید.</p>
</div>

In [ ]:
def effective_ids(tokenizer, text, limit):
    # TODO: متن کامل و ورودی واقعی forward
    return None

In [ ]:
def test_exercise():
    before = list(tokenizer.id_to_token)
    result = effective_ids(tokenizer,text,5)
    if result is None:
        return False
    full,context = result
    assert full==tokenizer.encode(text) and context==full[-5:]
    assert 0 in full
    assert effective_ids(tokenizer,'مدل',20)==(tokenizer.encode('مدل'),tokenizer.encode('مدل'))
    assert effective_ids(tokenizer,'مدل',1)[1]==tokenizer.encode('ل')
    assert tokenizer.id_to_token==before
    return True
exercise_complete = test_exercise()
print('PASS' if exercise_complete else 'INCOMPLETE: effective_ids')

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">فقط یک عامل را تغییر دهید</h2>
<p style="text-align:right">فقط پیشوندی را عوض کنید که بیرون <bdi dir="ltr">Context</bdi> است؛ پنج شناسهٔ آخر در هر دو <bdi dir="ltr">Prompt</bdi> ثابت بمانند.</p>
</div>

In [ ]:
a = torch.tensor([tokenizer.encode('مدل زبان مدل')])
b = torch.tensor([tokenizer.encode('زبان زبان مدل')])
assert torch.equal(a[:,-5:],b[:,-5:])
print('next IDs:',model.generate(a,1,greedy=True)[0,-1].item(),model.generate(b,1,greedy=True)[0,-1].item())

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">خرابی را پیدا کنید</h2>
<p style="text-align:right">ساختن شناسهٔ تازه پس از آموزش، جدول <bdi dir="ltr">Embedding</bdi> را بزرگ نمی‌کند. <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">encode_with_unknowns(tokenizer,text)</code> باید همان <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">encode</code> معتبر را همراه تعداد صفرها برگرداند؛ برای متن خالی <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">ValueError</code> بدهد.</p>
</div>

In [ ]:
try:
    model(torch.tensor([[tokenizer.vocab_size]]))
except ValueError as error:
    print('expected invalid new ID:',error)
else:
    raise AssertionError('out-of-vocabulary ID must be rejected')

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">اصلاح را خودتان بنویسید</h2>
<p style="text-align:right">علت را توضیح دهید، سپس تابع زیر را کامل کنید. خطای عمدی بالا یک نمونهٔ آموزشی است؛ آزمون پایین باید اصلاح شما را بسنجد.</p>
</div>

In [ ]:
def encode_with_unknowns(tokenizer, text):
    # TODO: اطلاعات ناشناخته را گزارش کنید، نه اینکه واژگان را عوض کنید
    return None

In [ ]:
def test_repair():
    result = encode_with_unknowns(tokenizer,'مدل X')
    if result is None:
        return False
    assert result==(tokenizer.encode('مدل X'),1)
    assert encode_with_unknowns(tokenizer,'مدل')[1]==0
    assert encode_with_unknowns(tokenizer,'XY')[1]==2
    try:
        encode_with_unknowns(tokenizer,'')
    except ValueError:
        pass
    else:
        raise AssertionError('empty prompt must be rejected')
    return True
repair_complete = test_repair()
print('PASS' if repair_complete else 'INCOMPLETE: encode_with_unknowns')

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">در <bdi dir="ltr">Mini-GPT</bdi> کجا به کار می‌آید؟</h2>
<p style="text-align:right">این همان <bdi dir="ltr">Tokenizer</bdi> و <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">generate</code> پروژه است. متن برگشتی کامل می‌ماند ولی <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">forward</code> فقط پنجرهٔ آخر را می‌بیند؛ موقعیت‌ها در آن پنجره از صفر شروع می‌شوند.</p>
</div>

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">با زبان خودتان توضیح دهید</h2>
<p style="text-align:right">وقتی <bdi dir="ltr">Prefix</bdi> از <bdi dir="ltr">Context</bdi> خارج شده، آیا تغییر <bdi dir="ltr">Temperature</bdi> راهی برای بازیابی آن است؟</p>
</div>
<div dir="rtl" style="text-align:right"><p style="text-align:right">پیش‌بینی و مشاهدهٔ من: …</p><p style="text-align:right">علت خرابی و اصلاح من: …</p></div>

<div dir="rtl" style="text-align:right"><p style="text-align:right"><a target="_self" href="http://127.0.0.1:8000/part-09/chapter-03/57-prompts.html">بازگشت به درس و ادامهٔ مسیر</a> · <a target="_self" href="http://127.0.0.1:8000/answers/57-prompts.html#lab-solution">فقط پس از تلاش: راه‌حل مرجع آزمایشگاه</a></p></div>